In [ ]:
"""Split gambar sumber, buat moving-window patch, dan simpan manifest."""

from pathlib import Path
import re

import cv2
import pandas as pd
from sklearn.model_selection import train_test_split


# ============================================================
# 1. Konfigurasi umum
# ============================================================

KAGGLE_INPUT = Path("/kaggle/input")
IS_KAGGLE = KAGGLE_INPUT.is_dir()

IMAGE_FOLDER_NAME = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih")
LABELS_FILENAME = Path(r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv\gabungan_dengan_jumlah_sel.csv")

IMAGE_COLUMN = "Nama_Gambar"
TARGET_COLUMN = "Jumlah Sel (sel/mL)"

WINDOW_SIZE = 32
STRIDE = 20

TEST_RATIO = 0.20
SEED = 42

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}


# ============================================================
# 2. Tentukan input dan output untuk Kaggle atau lokal
# ============================================================

if IS_KAGGLE:
    # Kaggle menyediakan input di /kaggle/input dan output di /kaggle/working.
    image_dirs = [
        path for path in KAGGLE_INPUT.rglob(IMAGE_FOLDER_NAME)
        if path.is_dir()
    ]
    label_files = list(KAGGLE_INPUT.rglob(LABELS_FILENAME))

    if len(image_dirs) != 1:
        raise FileNotFoundError(
            f"Harus ditemukan tepat satu folder '{IMAGE_FOLDER_NAME}' "
            f"di {KAGGLE_INPUT}; ditemukan {len(image_dirs)}."
        )

    if len(label_files) != 1:
        raise FileNotFoundError(
            f"Harus ditemukan tepat satu file '{LABELS_FILENAME}' "
            f"di {KAGGLE_INPUT}; ditemukan {len(label_files)}."
        )

    IMAGE_DIR = image_dirs[0]
    LABELS_CSV = label_files[0]
    OUTPUT_DIR = Path("/kaggle/working/CNN_Moving_Window")

else:
    # Path input lokal Windows.
    IMAGE_DIR = Path(
        r"D:\00 Kuliah\TA\Experiment\Dataset\Data Gambar\gambar_terpilih"
    )
    LABELS_CSV = Path(
        r"D:\00 Kuliah\TA\Experiment\Dataset\Data Sensor Kultivasi\csv"
        r"\gabungan_dengan_jumlah_sel.csv"
    )

    # Folder output lokal. Semua skrip tahap berikutnya sebaiknya
    # menggunakan folder output yang sama.
    OUTPUT_DIR = Path(
        r"D:\00 Kuliah\TA\Experiment\Model"
        r"\Convolutional neural network regression for low-cost microalgal density estimation"
        r"\CNN_Moving_Window"
    )


PATCH_DIR = OUTPUT_DIR / "patches"
MANIFEST_PATH = OUTPUT_DIR / "moving_window_manifest.csv"

if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f"Folder gambar tidak ditemukan: {IMAGE_DIR}")

if not LABELS_CSV.is_file():
    raise FileNotFoundError(f"CSV label tidak ditemukan: {LABELS_CSV}")

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Mode: {'Kaggle' if IS_KAGGLE else 'Lokal'}")
print(f"Folder gambar: {IMAGE_DIR}")
print(f"CSV label: {LABELS_CSV}")
print(f"Folder patch: {PATCH_DIR}")
print(f"Manifest: {MANIFEST_PATH}")


# ============================================================
# 3. Fungsi pencocokan gambar dan target
# ============================================================

def image_key(value) -> str:
    """Ambil nama gambar tanpa folder dan ekstensi."""
    normalized = str(value).replace("\\", "/")
    return Path(normalized).stem.strip().lower()


def parse_target(value) -> float:
    """Baca target angka, termasuk format ribuan seperti 45.577.100."""
    text = str(value).strip().replace(" ", "")

    # Contoh format ribuan: 45.577.100 atau 45,577,100.
    if re.fullmatch(r"\d{1,3}(?:[.,]\d{3})+", text):
        text = text.replace(".", "").replace(",", "")

    return float(text)


# ============================================================
# 4. Baca CSV label dan cocokkan dengan gambar
# ============================================================

labels = pd.read_csv(
    LABELS_CSV,
    encoding="utf-8-sig",
    dtype={
        IMAGE_COLUMN: "string",
        TARGET_COLUMN: "string",
    },
)

required_columns = {IMAGE_COLUMN, TARGET_COLUMN}
missing_columns = required_columns - set(labels.columns)

if missing_columns:
    raise KeyError(
        f"Kolom label tidak ditemukan di CSV: {sorted(missing_columns)}"
    )

labels["_key"] = labels[IMAGE_COLUMN].map(image_key)

if labels["_key"].duplicated().any():
    duplicated = labels.loc[
        labels["_key"].duplicated(keep=False), IMAGE_COLUMN
    ].tolist()
    raise ValueError(
        "Nama gambar duplikat di CSV label setelah ekstensi dihapus: "
        f"{duplicated[:10]}"
    )

label_lookup = {
    row["_key"]: {
        "label_name": str(row[IMAGE_COLUMN]),
        "target": parse_target(row[TARGET_COLUMN]),
    }
    for _, row in labels.iterrows()
}


# ============================================================
# 5. Kumpulkan pasangan gambar-label valid
# ============================================================

image_paths = sorted(
    path
    for path in IMAGE_DIR.rglob("*")
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if not image_paths:
    raise FileNotFoundError(f"Tidak ada gambar ditemukan di: {IMAGE_DIR}")

records = []
unmatched_images = []

for image_path in image_paths:
    key = image_key(image_path.name)
    label_info = label_lookup.get(key)

    if label_info is None:
        unmatched_images.append(image_path.name)
        continue

    image = cv2.imread(str(image_path), cv2.IMREAD_COLOR)
    if image is None:
        print(f"Lewati, gambar tidak dapat dibaca: {image_path}")
        continue

    records.append({
        "path": image_path,
        "source_image": image_path.relative_to(IMAGE_DIR).as_posix(),
        "Nama_Gambar": label_info["label_name"],
        "target_density": label_info["target"],
    })

if unmatched_images:
    print(
        f"Peringatan: {len(unmatched_images)} gambar tidak memiliki label. "
        f"Contoh: {unmatched_images[:5]}"
    )

if len(records) < 3:
    raise ValueError(
        f"Hanya ada {len(records)} pasangan gambar-label valid. "
        "Minimal diperlukan 3 pasangan."
    )

print(f"Jumlah pasangan gambar-label valid: {len(records)}")


# ============================================================
# 6. Split gambar sumber sebelum membuat patch
# ============================================================

all_indices = list(range(len(records)))

train_indices, test_indices = train_test_split(
    all_indices,
    test_size=TEST_RATIO,
    random_state=SEED,
    shuffle=True,
)

split_indices = {
    "train": train_indices,
    "test": test_indices,
}

print(f"Gambar train: {len(train_indices)}")
print(f"Gambar test: {len(test_indices)}")


# ============================================================
# 7. Buat dan simpan moving-window patch
# ============================================================

manifest_rows = []
patch_counts = {split_name: 0 for split_name in split_indices}
source_patch_counts = {split_name: 0 for split_name in split_indices}

for split_name, source_indices in split_indices.items():
    for source_index in source_indices:
        record = records[source_index]

        image = cv2.imread(str(record["path"]), cv2.IMREAD_COLOR)
        if image is None:
            print(f"Lewati, gambar gagal dibaca ulang: {record['path']}")
            continue

        height, width = image.shape[:2]

        if width < WINDOW_SIZE or height < WINDOW_SIZE:
            print(
                f"Lewati, gambar lebih kecil dari window "
                f"{WINDOW_SIZE}x{WINDOW_SIZE}: {record['path'].name}"
            )
            continue

        patch_number = 0

        # Window bergerak dari kiri ke kanan, lalu atas ke bawah.
        # Sisa tepi yang tidak cukup untuk satu window penuh tidak dipadding.
        for y in range(0, height - WINDOW_SIZE + 1, STRIDE):
            for x in range(0, width - WINDOW_SIZE + 1, STRIDE):
                patch = image[y:y + WINDOW_SIZE, x:x + WINDOW_SIZE]
                patch_number += 1

                patch_name = (
                    f"{record['path'].stem}_y{y:04d}_x{x:04d}.png"
                )
                patch_path = (
                    PATCH_DIR
                    / split_name
                    / Path(record["source_image"]).parent
                    / patch_name
                )
                patch_path.parent.mkdir(parents=True, exist_ok=True)

                if not cv2.imwrite(str(patch_path), patch):
                    print(f"Gagal menyimpan patch: {patch_path}")
                    continue

                manifest_rows.append({
                    "patch_path": patch_path.relative_to(OUTPUT_DIR).as_posix(),
                    "source_image": record["source_image"],
                    "Nama_Gambar": record["Nama_Gambar"],
                    "split": split_name,
                    "patch_number": patch_number,
                    "x": x,
                    "y": y,
                    "window_width": WINDOW_SIZE,
                    "window_height": WINDOW_SIZE,
                    "source_width": width,
                    "source_height": height,
                    "target_density": record["target_density"],
                })

                patch_counts[split_name] += 1

        if patch_number > 0:
            source_patch_counts[split_name] += 1

        print(
            f"{record['path'].name}: {width}x{height}, "
            f"{patch_number} patch, subset={split_name}"
        )


# ============================================================
# 8. Simpan manifest dan ringkasan
# ============================================================

manifest = pd.DataFrame(manifest_rows)

if manifest.empty:
    raise ValueError(
        "Tidak ada patch yang berhasil dibuat. "
        f"Gambar harus minimal {WINDOW_SIZE}x{WINDOW_SIZE} px."
    )

manifest.to_csv(
    MANIFEST_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("\nRingkasan hasil:")
print(f"Gambar sumber valid: {len(records)}")
print(
    f"Gambar yang menghasilkan patch — "
    f"train: {source_patch_counts['train']}, "
    f"test: {source_patch_counts['test']}"
)
print(
    f"Jumlah patch — "
    f"train: {patch_counts['train']}, "
    f"test: {patch_counts['test']}"
)
print(f"Total patch: {sum(patch_counts.values())}")
print(f"Folder patch: {PATCH_DIR}")
print(f"Manifest: {MANIFEST_PATH}")